# 📘 NB_02_Load_UKWeatherData

> Fetch historical weather data (temperature only) for the top 5 most populated UK cities and store it as raw Delta tables.

---

### 📒 Notebook Summary  
**API-driven data ingestion using Open-Meteo with metadata logging in Microsoft Fabric**

📌 Step 1: Setup Parameters

- Identify:
  - `lakehouse_id`
  - `workspace_id`
  - `notebook_name`
- Set query dates:
  - From **first day of current month** to **current date**
- Define table and schema names:
  - Log Table: `log.monitoring_log`
  - City Tables: `raw.<city>_hourly_weather`
- Configure Open-Meteo API client with retry and caching.

🏙️ Step 2: Read City Data

- Read Delta table from:  `abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Tables/raw/city_locations`

  - Filter cities where `country = 'United Kingdom'`
  - Cast population as long, sort descending
  - Select **top 5 cities by population**

🌐 Step 3: Call Open-Meteo API

- For each city:
  - Use `lat`, `lng`, `city`, `country`
  - Request:
    - `hourly.temperature_2m`
  - Convert response to a Pandas DataFrame
  - Add metadata columns: `city`, `country`

💾 Step 4: Save to Raw Tables

- Convert to Spark DataFrame
- Write Delta file to:`abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Tables/raw/_hourly_weather`

- Mode: `overwrite`  
- Options: `mergeSchema = true`

🪵 Step 5: Write Log Entry

- For each city write:
  - `notebook_name`
  - `table_name`
  - `run_id`
  - `user_name`
  - `row_count`
  - `status`
  - `error_message` (if any)
- Append entry to:`abfss://<workspace_id>@onelake.dfs.fabric.microsoft.com/<lakehouse_id>/Tables/log/monitoring_log`


In [ ]:
# Check what's already available
import pkg_resources
installed_packages = [pkg.key for pkg in pkg_resources.working_set]

# Only install what's missing
required = {'openmeteo-requests', 'requests-cache', 'retry-requests'}
missing = required - set(installed_packages)

if missing:
    ! pip install {' '.join(missing)} --no-deps --quiet

In [ ]:
# -----------------------------------------------
# 📦 Library Imports
# -----------------------------------------------
import requests
import uuid
import datetime
import time
import pandas as pd
import requests_cache
from decimal import Decimal, ROUND_HALF_UP
from retry_requests import retry
import openmeteo_requests

from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType, DoubleType, DateType
)

import sempy.fabric as fabric

# -----------------------------------------------
# 📅 Fabric Metadata & Path Setup
# -----------------------------------------------
lakehouse_id = fabric.get_lakehouse_id()
workspace_id = fabric.get_workspace_id()
lakehouse_name_resolved = fabric.resolve_item_name(lakehouse_id)
notebook_name = fabric.resolve_item_name(fabric.get_artifact_id())

raw_schema = "raw"
log_schema = "log"
log_table_name = "monitoring_log"

# Construct ABFS paths
log_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{log_schema}/{log_table_name}"

# -----------------------------------------------
# 🌐 Setup API Client with Retry + Cache
# -----------------------------------------------
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# -----------------------------------------------
# 🪚 Utility Functions
# -----------------------------------------------
def remove_all_characters(value: str) -> str:
    return ''.join(e.lower() for e in value if e.isalnum())

def log_changes(
    notebook_name: str,
    table_name: str,
    log_path: str,
    row_before: int,
    row_after: int,
    row_inserted: int,
    user: str,
    operation_type: str,
    exec_time: float,
    message: str,
    error_message: str = ""
) -> None:
    try:
        run_id = str(uuid.uuid4())
        timestamp = datetime.utcnow()

        schema = StructType([
            StructField("notebook_name", StringType(), True),
            StructField("table_name", StringType(), True),
            StructField("run_id", StringType(), True),
            StructField("user_name", StringType(), True),
            StructField("operation_type", StringType(), True),
            StructField("row_count", IntegerType(), True),
            StructField("message", StringType(), True),
            StructField("error_message", StringType(), True),
            StructField("timestamp", TimestampType(), True)
        ])

        log_data = [{
            "notebook_name": notebook_name,
            "table_name": table_name,
            "run_id": run_id,
            "user_name": user,
            "operation_type": operation_type,
            "row_count": row_inserted,
            "message": message,
            "error_message": error_message,
            "timestamp": timestamp
        }]

        df_log = spark.createDataFrame(log_data, schema=schema)
        df_log.write.format("delta").mode("append").save(log_path)
        print(f"✅ Log written for {table_name}")
    except Exception as e:
        print(f"❌ Failed to write log: {str(e)}")

# -----------------------------------------------
# 🏠 Load Top 5 UK Cities
# -----------------------------------------------
df_cities = spark.read.format("delta").load(
    f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{raw_schema}/city_locations"
)

df_top5 = (
    df_cities
    .filter(col("country") == "United Kingdom")
    .withColumn("population_long", col("population").cast("long"))
    .orderBy(col("population_long").desc())
    .limit(5)
)

# Convert Spark to Pandas for looping
top5_cities_pd = df_top5.select("city", "lat", "lng", "country").toPandas()
print("\n📌 Top 5 UK Cities:")
display(top5_cities_pd)

# -----------------------------------------------
# 🔄 Loop to Fetch and Save Weather Data
# -----------------------------------------------
for _, row in top5_cities_pd.iterrows():
    city = row["city"]
    lat = row["lat"]
    lng = row["lng"]
    country = row["country"]

    city_table_name = f"{remove_all_characters(city)}_hourly_weather"
    qualified_table_name = f"{raw_schema}.{city_table_name}"
    target_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{raw_schema}/{city_table_name}"

    start_date = datetime.today().replace(day=1)
    end_date = datetime.now()

    print(f"\n🛡️ Fetching data for: {city} ({start_date.date()} → {end_date.date()})")
    time.sleep(10)

    try:
        responses = openmeteo.weather_api(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": lat,
                "longitude": lng,
                "start_date": start_date.strftime('%Y-%m-%d'),
                "end_date": end_date.strftime('%Y-%m-%d'),
                "hourly": ["temperature_2m"]
            }
        )
        hourly = responses[0].Hourly()

        df_pd = pd.DataFrame({
            "date": pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq=pd.Timedelta(seconds=hourly.Interval()),
                inclusive="left"
            ),
            "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
            "city": city,
            "country": country
        })

        df_spark = spark.createDataFrame(df_pd)
        row_inserted = df_spark.count()

        df_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(target_path)
        df_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(qualified_table_name)

        log_changes(
            notebook_name=notebook_name,
            table_name=qualified_table_name,
            log_path=log_path,
            row_before=0,
            row_after=row_inserted,
            row_inserted=row_inserted,
            user="prathy.k",
            operation_type="api_load",
            exec_time=0,
            message=f"{city} weather data loaded"
        )
    except Exception as e:
        print(f"❌ Failed to load data for {city}: {str(e)}")
        log_changes(
            notebook_name=notebook_name,
            table_name=qualified_table_name,
            log_path=log_path,
            row_before=0,
            row_after=0,
            row_inserted=0,
            user="prathy.k",
            operation_type="api_load",
            exec_time=0,
            message="API load failed",
            error_message=str(e)
        )
